# Smogon Data Cleaning

## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

## Global Variables

In [2]:
DATA_PATH   = Path(Path.cwd().parent.parent, 'data')

BRONZE_PATH = Path(DATA_PATH / '1-bronze')
SILVER_PATH = Path(DATA_PATH / '2-silver')

SMOGON_PATH        = Path(BRONZE_PATH / "Smogon")
SILVER_SMOGON_PATH = Path(SILVER_PATH / "Smogon")

CATEGORIES = ['anythinggoes', 'ubers', 'ubersuu', 'ou', 'uu', 'ru', 'nu', 'pu', 'zu']

KEY_COLS = ['date', 'category', 'rating']

SAMPLE_USAGE_PATH = Path(SMOGON_PATH / '2026' / '08' / 'gen9ou-1500.json')

WEIGHT_LIST = pa.list_(pa.struct([('name', pa.string()), ('weight', pa.float64())]))
CHECKS_LIST = pa.list_(pa.struct([('name', pa.string()), ('n', pa.float64()), ('p', pa.float64()), ('d', pa.float64())]))

## Helpers

In [3]:
def normalize_json_column(df:pd.DataFrame, column_name: str) -> pd.DataFrame:
    try:
        new_columns = pd.json_normalize(df[column_name].to_list())
        for i in new_columns:
            df[f"{column_name}_{i}"] = new_columns[i]

        df = df.drop(columns=column_name)
        
        return df

    except KeyError:
        print(f"Non Existing Column with name: {column_name}")
        return df


In [4]:
def smogon_file_key(file_path: Path) -> dict:
    category, rating = file_path.stem.removeprefix('gen9').rsplit('-', 1)
    date = pd.Timestamp(int(file_path.parent.parent.name), int(file_path.parent.name), 1, tz='UTC')

    return {'date': date, 'category': category, 'rating': int(rating)}

In [5]:
def create_metagame_dataframe(dir_path: Path) -> pd.DataFrame:
    data_list = []

    for file_path in tqdm(sorted(dir_path.glob('*/*/*.json'))):
        with open(file_path, 'r') as f:
            data = json.load(f)
            data_list.append(smogon_file_key(file_path) | {'info': data['info']})

    return normalize_json_column(pd.DataFrame(data_list), 'info')

In [6]:
def create_usage_dataframe(file_path: Path) -> pd.DataFrame:
    with open(file_path, 'r') as f:
        data = json.load(f)

    df = pd.DataFrame.from_dict(data['data'], orient='index').rename_axis('pokemon').reset_index()

    return df.assign(**smogon_file_key(file_path))

In [7]:
def weight_sum(series: pd.Series) -> pd.Series:
    rows = np.repeat(np.arange(len(series)), series.list.len().to_numpy())
    weights = series.list.flatten().struct.field('weight').to_numpy()

    return pd.Series(np.bincount(rows, weights=weights, minlength=len(series)), index=series.index)

## Creating DataFrames and Preparing the Data

### Metagame:

In [8]:
df = create_metagame_dataframe(SMOGON_PATH)
df = df.set_index(KEY_COLS)

print(f"N of Rows: {len(df)}")
df.head()

100%|██████████| 872/872 [00:32<00:00, 26.45it/s]

N of Rows: 872


info_team type  info_cutoff  \
date                      category     rating                               
2024-08-01 00:00:00+00:00 anythinggoes 0                None          0.0   
                                       1500             None       1500.0   
                                       1630             None       1630.0   
                                       1760             None       1760.0   
                          nu           0                None          0.0   

                                               info_cutoff deviation  \
date                      category     rating                          
2024-08-01 00:00:00+00:00 anythinggoes 0                           0   
                                       1500                        0   
                                       1630                        0   
                                       1760                        0   
                          nu           0                           0   

                                                  info_metagame  \
date                      category     rating                     
2024-08-01 00:00:00+00:00 anythinggoes 0       gen9anythinggoes   
                                       1500    gen9anythinggoes   
                                       1630    gen9anythinggoes   
                                       1760    gen9anythinggoes   
                          nu           0                 gen9nu   

                                               info_number of battles  
date                      category     rating                          
2024-08-01 00:00:00+00:00 anythinggoes 0                       203203  
                                       1500                    203203  
                                       1630                    203203  
                                       1760                    203203  
                          nu           0                        41001

In [9]:
print(df[['info_team type', 'info_cutoff deviation']].value_counts(dropna=False))

print("----------")
print(f"Metagame != gen9 + category: {(df['info_metagame'] != 'gen9' + df.index.get_level_values('category')).sum()}")
print(f"Cutoff != rating:            {(df['info_cutoff'] != df.index.get_level_values('rating')).sum()}")

info_team type  info_cutoff deviation
NaN             0                        872
Name: count, dtype: int64
----------
Metagame != gen9 + category: 0
Cutoff != rating:            0


In [10]:
# Recreate for not popping errors when running this cell 2+ times
df = create_metagame_dataframe(SMOGON_PATH)

df = df.drop(columns=['info_team type', 'info_cutoff deviation'])
df = df.drop(columns=['info_metagame', 'info_cutoff'])

df = df.rename(columns={'info_number of battles': 'n_battles'})

df = df.set_index(KEY_COLS)

print(f"N of Rows: {len(df)}")
df.head()

100%|██████████| 872/872 [00:33<00:00, 26.12it/s]

N of Rows: 872


n_battles
date                      category     rating           
2024-08-01 00:00:00+00:00 anythinggoes 0          203203
                                       1500       203203
                                       1630       203203
                                       1760       203203
                          nu           0           41001

In [11]:
# Data quality checks
print(f"Duplicated keys: {df.index.duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())

# The ratings are weighted views of the same battles, so a month-category has 4 files with the same number of battles
per_month = df['n_battles'].groupby(['date', 'category'])

print("----------")
print(f"Categories not in CATEGORIES:            {(~df.index.get_level_values('category').isin(CATEGORIES)).sum()}")
print(f"Number of battles <= 0:                  {df['n_battles'].le(0).sum()}")

Duplicated keys: 0
----------
Nulls per column:
n_battles    0
dtype: int64
----------
Categories not in CATEGORIES:            0
Number of battles <= 0:                  0


In [12]:
# Data types: rating is an index level, so the index is rebuilt
df = df.reset_index().astype({'rating': 'int16', 'n_battles': 'int32'}).set_index(KEY_COLS)

print(df.dtypes)
print(f"----------\nIndex dtypes:\n{df.index.dtypes}")

n_battles    int32
dtype: object
----------
Index dtypes:
date        datetime64[us, UTC]
category                    str
rating                    int16
dtype: object


In [13]:
keys = df.index.to_frame(index=False)
pd.crosstab(keys['category'], keys['rating'])

rating,0,1500,1630,1695,1760,1825
category,,,,,,
anythinggoes,25,25,25,0,25,0
nu,25,25,25,0,25,0
ou,25,25,0,25,0,25
pu,25,25,25,0,25,0
ru,25,25,25,0,25,0
ubers,25,25,25,0,25,0
ubersuu,18,18,18,0,18,0
uu,25,25,25,0,25,0
zu,25,25,25,0,25,0


In [14]:
# Battles per month
df.xs(0, level='rating')['n_battles'].unstack('category')

category,anythinggoes,nu,ou,pu,ru,ubers,ubersuu,uu,zu
date,,,,,,,,,
2024-08-01 00:00:00+00:00,203203.0,41001.0,1141008.0,42983.0,51901.0,169039.0,4449.0,114185.0,16869.0
2024-09-01 00:00:00+00:00,204062.0,31864.0,1093463.0,40986.0,45932.0,161343.0,2861.0,108940.0,15030.0
2024-10-01 00:00:00+00:00,222027.0,33537.0,1137912.0,38134.0,55518.0,169755.0,5248.0,122488.0,13928.0
2024-11-01 00:00:00+00:00,214643.0,26403.0,995612.0,35747.0,47403.0,150762.0,2612.0,100718.0,12930.0
2024-12-01 00:00:00+00:00,206390.0,27067.0,1005497.0,29987.0,43006.0,140337.0,2496.0,85126.0,9597.0
2025-01-01 00:00:00+00:00,194272.0,31968.0,1086636.0,36633.0,48304.0,140601.0,1585.0,90110.0,15323.0
2025-02-01 00:00:00+00:00,193308.0,29282.0,1006315.0,31071.0,47152.0,132762.0,2416.0,95500.0,11903.0
2025-03-01 00:00:00+00:00,248366.0,32129.0,1253206.0,33275.0,55224.0,172613.0,2732.0,115385.0,12562.0
2025-04-01 00:00:00+00:00,232678.0,30637.0,1137058.0,39101.0,61386.0,170234.0,1019.0,112727.0,13878.0


In [15]:
SILVER_SMOGON_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_SMOGON_PATH / 'metagame.parquet', index=False)

check = pd.read_parquet(SILVER_SMOGON_PATH / 'metagame.parquet').set_index(KEY_COLS)
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same index dtypes: {check.index.dtypes.equals(df.index.dtypes)} | Same data: {check.equals(df)}")

Rows: 872 | Same dtypes: True | Same index dtypes: True | Same data: True


### Usage:

In [16]:
df = create_usage_dataframe(SAMPLE_USAGE_PATH)
df = df.set_index(KEY_COLS + ['pokemon'])

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 397


Raw count  \
date                      category rating pokemon                 
2026-08-01 00:00:00+00:00 ou       1500   Great Tusk     439754   
                                          Gholdengo      351131   
                                          Kingambit      321215   
                                          Dragonite      252204   
                                          Zamazenta      229217   

                                                        Viability Ceiling  \
date                      category rating pokemon                           
2026-08-01 00:00:00+00:00 ou       1500   Great Tusk  [25395, 91, 84, 65]   
                                          Gholdengo   [21668, 91, 84, 65]   
                                          Kingambit   [21066, 91, 84, 65]   
                                          Dragonite   [17033, 91, 84, 66]   
                                          Zamazenta   [14704, 91, 85, 68]   

                                                                                              Abilities  \
date                      category rating pokemon                                                         
2026-08-01 00:00:00+00:00 ou       1500   Great Tusk              {'protosynthesis': 295626.6821556209}   
                                          Gholdengo                  {'goodasgold': 230642.13956819056}   
                                          Kingambit   {'supremeoverlord': 201396.796058544, 'pressur...   
                                          Dragonite   {'multiscale': 160459.07621344167, 'innerfocus...   
                                          Zamazenta              {'dauntlessshield': 162356.3128215272}   

                                                                                                  Items  \
date                      category rating pokemon                                                         
2026-08-01 00:00:00+00:00 ou       1500   Great Tusk  {'focussash': 541.4690324764917, 'heavydutyboo...   
                                          Gholdengo   {'airballoon': 84987.00040084495, 'heavydutybo...   
                                          Kingambit   {'leftovers': 109348.36228484588, 'blackglasse...   
                                          Dragonite   {'heavydutyboots': 106246.71947200227, 'choice...   
                                          Zamazenta   {'heavydutyboots': 24479.65098338705, 'leftove...   

                                                                                                Spreads  \
date                      category rating pokemon                                                         
2026-08-01 00:00:00+00:00 ou       1500   Great Tusk  {'Jolly:4/252/0/0/0/252': 33576.89885893685, '...   
                                          Gholdengo   {'Timid:4/0/0/252/0/252': 27501.347460889723, ...   
                                          Kingambit   {'Adamant:112/252/0/0/0/144': 4640.47654602892...   
                                          Dragonite   {'Jolly:0/252/0/0/4/252': 15385.342137548363, ...   
                                          Zamazenta   {'Jolly:0/252/0/0/4/252': 30620.74591685913, '...   

                                                                                                  Moves  \
date                      category rating pokemon                                                         
2026-08-01 00:00:00+00:00 ou       1500   Great Tusk  {'closecombat': 102467.01983542595, 'headlongr...   
                                          Gholdengo   {'thunderbolt': 18992.576196275924, 'powergem'...   
                                          Kingambit   {'kowtowcleave': 153758.49705357171, 'ironhead...   
                                          Dragonite   {'dragondance': 124085.1088365815, 'extremespe...   
                                          Zamazenta   {'closecombat': 77559.22540329775, 'crunch': 1...   

                                            

Viability Ceiling: [players, top GXE, 99th percentile GXE, 95th percentile GXE]

GXE (Glicko X-Act Estimate) is Pokémon Showdown's estimated chance (%) that a player beats a random opponent from the ladder. For each Pokémon, Smogon ranks the players who used it by GXE. 

E.g. Great Tusk (2026-08, OU-1500) [25395, 91, 84, 65]: 25395 players used it, the best has GXE 91, the top 1% have GXE ≥ 84 and the top 5% GXE ≥ 65.

In [17]:
# Names that are not Showdown ids (lowercase letters and digits)
for col in ['Abilities', 'Items', 'Moves', 'Tera Types']:
    names = pd.Series([name for d in df[col] for name in d]).drop_duplicates()
    print(f"{col}: {names[~names.str.fullmatch(r'[a-z0-9]+')].to_list()}")

print("----------")
print(f"Viability Ceiling lengths:   {df['Viability Ceiling'].map(len).value_counts().to_dict()}")

# Checks and Counters values changed format in 2026-03, an older file is read to compare
older = create_usage_dataframe(SMOGON_PATH / '2026' / '02' / 'gen9ou-1500.json')

for month, sample in [('2026-02', older), ('2026-08', df)]:
    values = pd.Series([v for d in sample['Checks and Counters'] for v in d.values()])
    print(f"Checks and Counters in {month}: {values.map(lambda v: type(v).__name__).value_counts().to_dict()} | lengths: {values.map(len).value_counts().to_dict()} | e.g. {values.iloc[0]}")

Abilities: []
Items: []
Moves: ['']
Tera Types: []
----------
Viability Ceiling lengths:   {4: 397}
Checks and Counters in 2026-02: {'list': 30528} | lengths: {3: 30528} | e.g. [20800.84620698942, 0.5510525551388569, 0.0034486868480633867]
Checks and Counters in 2026-08: {'dict': 25402} | lengths: {3: 25402} | e.g. {'n': 2162.0035088754353, 'p': 0.45180431277148453, 'd': 0.010703228475371517}


In [18]:
# Recreate for not popping errors when running this cell 2+ times
weight_cols    = ['abilities', 'items', 'moves', 'spreads', 'tera_types', 'teammates']
viability_cols = ['viability_players', 'viability_top_gxe', 'viability_99th_gxe', 'viability_95th_gxe']

# One file at a time due to memory issues
frames = []

for file_path in tqdm(sorted(SMOGON_PATH.glob('*/*/*.json'))):
    df = create_usage_dataframe(file_path)

    if df.empty:
        continue

    df = df.drop(columns='Happiness')
    df = df.rename(columns={'Raw count': 'raw_count', 'Abilities': 'abilities', 'Items': 'items', 'Moves': 'moves', 'Spreads': 'spreads', 'Tera Types': 'tera_types', 'Teammates': 'teammates', 'Checks and Counters': 'checks_and_counters'})

    df[viability_cols] = pd.DataFrame(df['Viability Ceiling'].to_list(), index=df.index)
    df = df.drop(columns='Viability Ceiling')

    # An empty move slot '' renamed to 'nothing' like the Items
    for col in weight_cols:
        df[col] = pd.Series(df[col].map(lambda d: [{'name': k or 'nothing', 'weight': v} for k, v in d.items()]), dtype=pd.ArrowDtype(WEIGHT_LIST))

    # A check is [n, p, d] until 2026-02 and {'n', 'p', 'd'} since 2026-03
    df['checks_and_counters'] = pd.Series(df['checks_and_counters'].map(lambda d: [{'name': k} | (v if isinstance(v, dict) else dict(zip(['n', 'p', 'd'], v))) for k, v in d.items()]), dtype=pd.ArrowDtype(CHECKS_LIST))

    frames.append(df)

df = pd.concat(frames, ignore_index=True)
del frames

df = df.set_index(KEY_COLS + ['pokemon'])[['raw_count', 'usage'] + viability_cols + weight_cols + ['checks_and_counters']]

print(f"N of Rows: {len(df)}")
df.head()

100%|██████████| 872/872 [01:51<00:00,  7.85it/s]


N of Rows: 282145


raw_count  \
date                      category     rating pokemon                   
2024-08-01 00:00:00+00:00 anythinggoes 0      Iron Valiant      12792   
                                              Magikarp           1771   
                                              Duosion             122   
                                              Eevee               597   
                                              Torracat             76   

                                                               usage  \
date                      category     rating pokemon                  
2024-08-01 00:00:00+00:00 anythinggoes 0      Iron Valiant  0.025440   
                                              Magikarp      0.003423   
                                              Duosion       0.000263   
                                              Eevee         0.001164   
                                              Torracat      0.000145   

                                                            viability_players  \
date                      category     rating pokemon                           
2024-08-01 00:00:00+00:00 anythinggoes 0      Iron Valiant               1414   
                                              Magikarp                    142   
                                              Duosion                      12   
                                              Eevee                       114   
                                              Torracat                     17   

                                                            viability_top_gxe  \
date                      category     rating pokemon                           
2024-08-01 00:00:00+00:00 anythinggoes 0      Iron Valiant                 83   
                                              Magikarp                     74   
                                              Duosion                      60   
                                              Eevee                        67   
                                              Torracat                     63   

                                                            viability_99th_gxe  \
date                      category     rating pokemon                            
2024-08-01 00:00:00+00:00 anythinggoes 0      Iron Valiant                  75   
                                              Magikarp                      67   
                                              Duosion                       60   
                                              Eevee                         63   
                                              Torracat                      63   

                                                            viability_95th_gxe  \
date                      category     rating pokemon                            
2024-08-01 00:00:00+00:00 anythinggoes 0      Iron Valiant                  59   
                                              Magikarp                      51   
                                              Duosion                       50   
                                              Eevee                         50   
                                              Torracat                      53   

                                                                                                    abilities  \
date                      category     rating pokemon                                                           
2024-08-01 00:00:00+00:00 anythinggoes 0      Iron Valiant        [{'name': 'quarkdrive', 'weight': 12792.0}]   
                                              Magikarp      [{'name': 'swiftswim', 'weight': 1106.0}
 {'na...   
                                              Duosion       [{'name': 'regenerator', 'weight': 17.0}
 {'na...   
                                              Eevee         [{'name': 'anticipation', 'weight': 105.0}
 {'...   
                                              Torracat      [{'name': 'intimida

In [20]:
# Data quality checks
nested_cols = weight_cols + ['checks_and_counters']

print(f"Duplicated keys: {df.index.duplicated().sum()}")
print(f"Files:           {df.groupby(KEY_COLS).ngroups}")

print("----------\nNulls per column:")
print(df.isna().sum())

print("----------\nEmpty lists per column:")
print(df[nested_cols].apply(lambda c: c.list.len().eq(0)).sum())

Duplicated keys: 0
Files:           870
----------
Nulls per column:
raw_count              0
usage                  0
viability_players      0
viability_top_gxe      0
viability_99th_gxe     0
viability_95th_gxe     0
abilities              0
items                  0
moves                  0
spreads                0
tera_types             0
teammates              0
checks_and_counters    0
dtype: int64
----------
Empty lists per column:
abilities                   0
items                       0
moves                       0
spreads                     0
tera_types                  0
teammates                   0
checks_and_counters    163697
dtype: uint64[pyarrow]


In [21]:
# Distinct names of each nested column
names = {col: pd.Series(df[col].list.flatten().struct.field('name').unique()) for col in nested_cols}

valid_tera = {'normal', 'fighting', 'flying', 'poison', 'ground', 'rock', 'bug', 'ghost', 'steel', 'fire', 'water', 'grass', 'electric', 'psychic', 'ice', 'dragon', 'dark', 'fairy', 'stellar'}
pokemon = set(df.index.get_level_values('pokemon'))

for col in ['abilities', 'items', 'moves', 'tera_types']:
    not_id = names[col][~names[col].str.fullmatch(r'[a-z0-9]+')]
    print(f"{col + ' not [a-z0-9]+:':<28} {len(not_id)} -> {not_id.to_list()}")

print("----------")
print(f"Columns with 'nothing':       {[col for col in nested_cols if names[col].eq('nothing').any()]}")
print(f"Tera types outside the 19:    {(~names['tera_types'].isin(valid_tera)).sum()}")
# 'empty' is a team with less than 6 Pokémon, the rest are too rare to have their own row
not_pokemon = names['teammates'][~names['teammates'].isin(pokemon)]
print(f"Teammates not a pokemon:      {len(not_pokemon)} -> {not_pokemon.to_list()}")
print(f"Checks not a pokemon:         {(~names['checks_and_counters'].isin(pokemon)).sum()}")

# Spreads are "Nature:hp/atk/def/spa/spd/spe" EVs
spreads = names['spreads'].str.extract(r'^(?P<nature>[A-Z][a-z]+):(?P<hp>\d+)/(?P<atk>\d+)/(?P<def>\d+)/(?P<spa>\d+)/(?P<spd>\d+)/(?P<spe>\d+)$')
evs = spreads.drop(columns='nature').dropna().astype(int)
over_252 = evs.gt(252).any(axis=1)

print("----------")
print(f"Distinct spreads:             {len(spreads)}")
print(f"Spreads not in the format:    {spreads['nature'].isna().sum()}")
print(f"Distinct natures:             {spreads['nature'].nunique()}")
print(f"Spreads with an EV > 252:     {over_252.sum()} -> {names['spreads'][over_252.index[over_252]].to_list()}")
print(f"Spreads with EV total > 510:  {evs.sum(axis=1).gt(510).sum()}")

abilities not [a-z0-9]+:     0 -> []
items not [a-z0-9]+:         0 -> []
moves not [a-z0-9]+:         0 -> []
tera_types not [a-z0-9]+:    0 -> []
----------
Columns with 'nothing':       ['items', 'moves']
Tera types outside the 19:    0
Teammates not a pokemon:      1 -> ['empty']
Checks not a pokemon:         0
----------
Distinct spreads:             2302386
Spreads not in the format:    0
Distinct natures:             25
Spreads with an EV > 252:     99 -> ['Naive:0/0/0/255/0/255', 'Timid:0/0/0/255/0/255', 'Modest:1/0/0/255/0/252', 'Hasty:0/1/0/252/0/255', 'Adamant:252/255/0/0/1/0', 'Hardy:0/0/0/255/0/252', 'Careful:252/1/0/0/255/0', 'Timid:0/0/0/252/1/255', 'Timid:0/0/0/252/0/255', 'Sassy:254/0/12/0/240/0', 'Hardy:255/50/0/120/0/38', 'Timid:4/0/0/252/0/253', 'Adamant:1/255/0/0/0/252', 'Modest:252/0/4/254/0/0', 'Adamant:0/255/0/0/0/255', 'Modest:0/0/0/255/0/255', 'Modest:130/0/0/255/0/125', 'Modest:0/0/124/254/132/0', 'Quiet:252/0/4/253/0/0', 'Modest:252/0/4/253/0/0', 'Modest:252

In [31]:
print("Weights per column:")
for col in weight_cols:
    weights = df[col].list.flatten().struct.field('weight')
    print(f"{col:<11} entries: {len(weights):>10} | negative: {weights.lt(0).sum():>3} | zero: {weights.eq(0).sum():>9}")

checks = df['checks_and_counters'].list.flatten()

# Usage is the share of teams, so it sums to 6 in each file
usage_sum = df['usage'].groupby(KEY_COLS).sum()
gxe = df[['viability_top_gxe', 'viability_99th_gxe', 'viability_95th_gxe']]

print("----------")
print(f"Usage out of (0, 1]:          {(~df['usage'].between(0, 1, inclusive='right')).sum()}")
print(f"Files with usage sum out of 5.9..6.01: {(~usage_sum.between(5.9, 6.01)).sum()} -> {usage_sum[~usage_sum.between(5.9, 6.01)].to_dict()}")
print(f"Raw count differing between ratings: {df['raw_count'].groupby(['date', 'category', 'pokemon']).nunique().gt(1).sum()}")

Weights per column:
abilities   entries:     580259 | negative:   0 | zero:      7732
items       entries:    6605171 | negative:   0 | zero:    639364
moves       entries:   10664110 | negative:   0 | zero:    613940
spreads     entries:   41067499 | negative:   0 | zero:   6537402
tera_types  entries:    3331933 | negative:   0 | zero:    131496
teammates   entries:   61130573 | negative:   0 | zero:   4488739
----------
Usage out of (0, 1]:          0
Files with usage sum out of 5.9..6.01: 28 -> {(Timestamp('2024-08-01 00:00:00+0000', tz='UTC'), 'anythinggoes', 0): 5.8238349, (Timestamp('2024-08-01 00:00:00+0000', tz='UTC'), 'ubers', 0): 5.8976857, (Timestamp('2024-09-01 00:00:00+0000', tz='UTC'), 'anythinggoes', 0): 5.8369646, (Timestamp('2024-09-01 00:00:00+0000', tz='UTC'), 'ubers', 0): 5.8984541, (Timestamp('2024-10-01 00:00:00+0000', tz='UTC'), 'anythinggoes', 0): 5.8577723, (Timestamp('2024-11-01 00:00:00+0000', tz='UTC'), 'anythinggoes', 0): 5.8510662, (Timestamp('2024-12-01 

In [32]:
# Abilities, items, spreads and tera types are one per Pokémon, moves are 4 (empty slots included)
totals = df[weight_cols].apply(weight_sum)
ratios = totals.drop(columns='abilities').div(totals['abilities'], axis=0)

print(f"Rows with abilities total 0: {totals['abilities'].eq(0).sum()}")
ratios.describe()

Rows with abilities total 0: 0


,items,moves,spreads,tera_types,teammates
count,2.821450e+05,2.821450e+05,2.821450e+05,2.821450e+05,282145.000000
mean,1.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,4.648335
std,4.045659e-15,1.108079e-14,5.172341e-15,4.539697e-15,0.346463
min,1.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,0.000000
25%,1.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,4.546638
50%,1.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,4.732863
75%,1.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,4.848923
max,1.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,5.073072


In [ ]:
gxe_cols = ['viability_top_gxe', 'viability_99th_gxe', 'viability_95th_gxe']

df = df.reset_index().astype({'rating': 'int16', 'raw_count': 'int32', 'viability_players': 'int32'} | dict.fromkeys(gxe_cols, 'int8')).set_index(KEY_COLS + ['pokemon'])

print(df.dtypes)
print(f"----------\nIndex dtypes:\n{df.index.dtypes}")

raw_count                                                          int32
usage                                                            float64
viability_players                                                  int32
viability_top_gxe                                                   int8
viability_99th_gxe                                                  int8
viability_95th_gxe                                                  int8
abilities              list<item: struct<name: string, weight: double...
items                  list<item: struct<name: string, weight: double...
moves                  list<item: struct<name: string, weight: double...
spreads                list<item: struct<name: string, weight: double...
tera_types             list<item: struct<name: string, weight: double...
teammates              list<item: struct<name: string, weight: double...
checks_and_counters    list<item: struct<name: string, n: double, p: ...
dtype: object
----------
Index dtypes:
date        

In [34]:
keys = df.index.to_frame(index=False)
pd.crosstab(keys['category'], keys['rating'])

rating,0,1500,1630,1695,1760,1825
category,,,,,,
anythinggoes,16395,14396,11863,0,8386,0
nu,8234,7452,6163,0,5023,0
ou,11254,10120,0,7710,0,6056
pu,8047,7172,6025,0,4972,0
ru,9064,8263,6760,0,5409,0
ubers,14064,12414,9407,0,5994,0
ubersuu,5100,4866,2158,0,1854,0
uu,10107,9171,7347,0,5585,0
zu,7788,6774,5751,0,5001,0


In [ ]:
# Pokemon
df.groupby(KEY_COLS).size().groupby('category').describe()

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
anythinggoes,100.0,510.400000,120.983428,293.0,431.50,524.5,605.00,682.0
nu,100.0,268.720000,51.021303,158.0,219.50,266.5,310.00,350.0
ou,100.0,351.400000,81.816622,226.0,282.25,359.0,420.75,458.0
pu,100.0,262.160000,47.871191,168.0,221.50,264.0,300.50,348.0
ru,100.0,294.960000,57.223807,193.0,245.50,302.0,347.25,374.0
ubers,100.0,418.790000,124.365322,208.0,319.00,434.0,520.00,585.0
ubersuu,70.0,199.685714,120.655401,12.0,109.50,162.5,331.00,400.0
uu,100.0,322.100000,70.386738,201.0,256.00,331.0,380.00,420.0
zu,100.0,253.140000,44.351675,169.0,216.50,248.5,290.75,343.0


In [37]:
# Most used Pokémon in the latest OU-1500
latest = df.index.get_level_values('date').max()
df.xs((latest, 'ou', 1500), level=KEY_COLS).nlargest(10, 'usage')[['raw_count', 'usage', 'viability_top_gxe']]

,raw_count,usage,viability_top_gxe
pokemon,,,
Great Tusk,439754,0.321548,91
Gholdengo,351131,0.250992,91
Kingambit,321215,0.225954,91
Dragonite,252204,0.177114,91
Zamazenta,229217,0.176892,91
Ogerpon-Wellspring,235897,0.176326,91
Iron Valiant,224249,0.162187,91
Raging Bolt,196944,0.155516,92
Slowking-Galar,206732,0.152608,91


In [38]:
# Entries per nested list
df[nested_cols].apply(lambda c: c.list.len()).describe()

,abilities,items,moves,spreads,tera_types,teammates,checks_and_counters
count,282145.0,282145.0,282145.0,282145.0,282145.0,282145.0,282145.0
mean,2.056599,23.410555,37.796559,145.554587,11.809293,216.663676,21.74231
std,0.809835,18.712941,21.059491,291.179906,5.397981,140.185517,51.44356
min,1.0,1.0,2.0,1.0,1.0,1.0,0.0
25%,1.0,9.0,24.0,19.0,8.0,102.0,0.0
50%,2.0,19.0,37.0,52.0,12.0,192.0,0.0
75%,3.0,34.0,49.0,147.0,17.0,329.0,15.0
max,5.0,121.0,472.0,4905.0,19.0,640.0,462.0


In [ ]:
SILVER_SMOGON_PATH.mkdir(parents=True, exist_ok=True)

# Written without the pandas metadata: pandas can't parse its dtype of the nested lists, and pd.read_parquet fails on it
table = pa.Table.from_pandas(df.reset_index(), preserve_index=False)
pq.write_table(table.replace_schema_metadata(), SILVER_SMOGON_PATH / 'usage.parquet')

# Compare to see if all df was written correctly
check = pq.read_table(SILVER_SMOGON_PATH / 'usage.parquet').to_pandas(types_mapper=lambda t: pd.ArrowDtype(t) if pa.types.is_list(t) else None).set_index(KEY_COLS + ['pokemon'])
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same index dtypes: {check.index.dtypes.equals(df.index.dtypes)} | Same data: {check.equals(df)}")

# A plain pandas read of some columns, the way Gold reads it
ou = pd.read_parquet(SILVER_SMOGON_PATH / 'usage.parquet', columns=['date', 'rating', 'pokemon', 'usage'], filters=[('category', '==', 'ou')])
print(f"\npd.read_parquet of OU: {len(ou)} rows")

Rows: 282145 | Same dtypes: True | Same index dtypes: True | Same data: True

pd.read_parquet of OU: 35140 rows
